# [실습1] 양극소재 결정구조 데이터 분석 및 전처리
---

## 실습 목표

- Python 실행 환경와 실습 데이터를 확인합니다.
- 데이터를 간단하게 분석합니다.
- 결측 데이터, 이상 데이터 등에 대한 전처리를 수행합니다.
- 데이터를 학습/테스트/검증 데이터셋으로 구분합니다.

---

## 실습 목차

1. **환경 설정 및 데이터 로딩** 

2. **데이터 구성 요소 확인** 

3. **결측 데이터 식별**

4. **결측값 대체** 

5. **이상치 전처리** 

6. **실습 데이터 불러오기 및 전처리 알고리즘 적용**

7. **데이터 구분** 
---

## 실습 개요

이번 실습에서는 양극재 결정구조 분류 데이터셋의 구성 요소를 확인하고 간단한 분석을 해볼 예정입니다. 이를 위해 토이 데이터셋을 이용해 사전에 연습해본 후, 이를 그대로 실습 데이터에 적용해볼 예정입니다.

---

## 1. 환경 설정 및 데이터 로딩

---

### 1.1 필수 라이브러리 불러오기
데이터를 불러오고 분석하기 위해 사용해야하는 라이브러리들을 불러옵니다. 이번 실습이 아닌 추후 개인 컴퓨터에서 불러올 경우, 해당 라이브러리들의 설치가 필요합니다. 

In [ ]:
## 라이브러리 불러오기
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import scipy
import yellowbrick


각 라이브러리가 어떤 코드에서 사용되는지 정확히 파악하기 위해, 세부 함수들은 __그 함수를 사용할 때__ 불러올 것입니다.

---

### 1.2 토이 데이터 불러오기
양극재 결정구조 분류 데이터셋을 이용해 실습을 진행하기 전, 간단한 토이 데이터셋을 이용해 일반적인 데이터 전처리 과정이 어떻게 이뤄지는지 살펴볼 예정입니다.

토이 데이터셋은 인공지능을 입문할 때 가장 많이 사용하는 'titanic dataset'입니다. 

해당 데이터에는 타이타닉 사건 때 배에 있었던 승객들의 명단을 포함하고 있습니다.

pandas 라이브러리를 이용해 데이터를 불러옵니다.

In [ ]:
df = pd.read_csv('./data/titanic.csv')  # df: dataframe의 줄임말
print(df)

In [ ]:
# shape 기능을 이용하면 데이터의 차원을 확인할 수 있습니다.
print(df.shape)

### [TODO] 주석에서 요구하는대로 ____를 적절히 수정해보세요.

In [ ]:
# display 함수를 이용하면 표 형태로 데이터 확인이 가능합니다.
_______(df)

---

## 2. 데이터 구성 요소 확인
---
### 2.1 데이터 자료형 확인
데이터를 변수나 파일에 저장할 때, 크게 네 가지 형태 중 하나로 저장하게 됩니다:

1. __float__: 실수
2. __int__: 정수
3. __bool__: 참 혹은 거짓
4. __object__: 글자들 (현재는 string 타입을 object 타입으로 표시, 변환 가능)

이 네 가지 형태를 자료형(data type)이라고 하고, 프로그래밍 언어에서 흔히 __'dtype'__이라고 줄여서 씁니다.

In [ ]:
plt.figure()
df.dtypes.value_counts().plot.pie(ylabel='')
plt.title('Data Types')
plt.show()

이 중, object 자료형, 즉 글자들로 이뤄진 데이터는 인공지능 모델에 어떻게 활용할지 잘 고려해봐야 합니다.

글자 데이터 중, 모델의 예측 성능에 전혀 도움이 되지 않는 데이터도 있고, 도움이 되더라도 전처리를 거쳐야하는 경우가 많습니다.

### 2.2 데이터 통계량 확인
양극재 결정구조 분류 데이터셋에서 일반적인 통계량을 확인합니다. 흔히 확인하는 데이터 통계량은 다음과 같습니다:
- 평균(mean)
- 표준편차(std)
- 제 1 사분위수(25% quantile)
- 제 3 사분위수(75% quantile)
- 최소값(min)
- 최대값(max)

숫자 데이터인 int, float에 대해서만 통계량을 확인합니다.

In [ ]:
display(df.describe().T)

---

### 2.3 데이터 고유값 확인
데이터 항목 별 고유값의 갯수를 확인합니다. 고유값의 갯수에 따라 카테고리화(categorize)를 할지 미리 결정할 수 있습니다. 해당 내용은 아래에서 자세히 다룹니다.

In [ ]:
plt.figure()
df.nunique().plot.bar()
plt.title('Number of different values')
plt.show()

Passenger, Name, Ticket 데이터 항목은 전체 데이터 갯수에 비해 과하게 많은 고유값을 갖고 있다는 것을 확인할 수 있습니다.

데이터 항목 이름에서도 유추할 수 있듯이, 글자 데이터이기 때문에 생기는 대표적인 현상입니다.

---

## 3. 결측 데이터 식별
---
### 3.1 결측치 갯수 확인

결측치가 있는지 테스트합니다. 결측치는 missing data라고도 부릅니다.
#### 결측치란?
- 데이터베이스 구축 과정에서, 해당 항목에 속하지 않아서 데이터 자체를 비어두는 경우 등, 특정 이유 때문에 없는 데이터가 존재할 수 있습니다. 이러한 빈 데이터를 결측치(missing data)라고 합니다.
- 기본 Python 언어에서는 __None__ 이라고 표현하고, 특정 라이브러리에서는 __NaN__ 이라고 표현합니다. 
- 다른 언어에서는 __Null__ 등으로 표현되기도 합니다.

pandas 라이브러리의 isnull() 함수를 이용해 결측 데이터의 갯수를 확인할 수 있습니다.

In [ ]:
df.isnull().sum()

Age, Cabin, Embarked 데이터에 결측 데이터가 존재하는 것을 확인할 수 있습니다.

결측 데이터가 전체 데이터 항목 중 차지하는 비율을 아래와 같이 확인할 수 있습니다.

### [TODO] print 함수의 설명에서 요구하는대로 적절한 변수를 _____에 넣어보세요.

In [ ]:
isNaN = df.isna().sum().sort_values(ascending=True)
isNaN_ratio = isNaN/df.shape[0]
print("NaN 데이터 갯수: \n", _______)
print("\n")
print("NaN 데이터 비율: \n", ______________)

Embarded는 결측치가 2개만 존재하고, Age는 대략 20%의 결측치가 존재합니다.

또, Cabin 항목은 77%가 결측치인 것을 확인할 수 있습니다.

---

### 3.2 결측 데이터 항목 식별

만약 결측치가 특정 비율보다 많다면, 해당 데이터 항목은 학습에 사용하기 어렵습니다. 이러한 데이터 항목을 식별하고 학습 데이터에서 제거해야 합니다.

만약 결측 데이터 항목이 있다면 아래와 같이 자동으로 식별할 수 있습니다.

In [ ]:
def check_NaN(df, threshold_NaN):  
    isNaN_ratio = (df.isna().sum()/df.shape[0]).sort_values(ascending=True)
    drop_cols = []
    if isNaN_ratio.max() == 0.0:
        print("데이터프레임 안에 NaN 데이터가 없습니다.")
    else:
        drop_cols = np.array(isNaN_ratio[isNaN_ratio > threshold_NaN].index)
        print('NaN 데이터의 비율이', threshold_NaN*100,'% 를 넘는 데이터 형을 식별합니다. 데이터:', drop_cols)
    return drop_cols

threshold_NaN 변수는 사용자가 정의하는 임계값입니다. 결측치가 전체 데이터 중 30% 가 넘는다면 결측 데이터 항목으로 식별하겠습니다.

### [TODO] check_NaN 함수의 형태를 확인하고 적합한 변수를 ____에 넣어보세요.

In [ ]:
threshold_NaN = 0.3
drop_cols = check_NaN(df, ______________)

---

### 3.3 결측 데이터 항목 제거
결측 데이터 항목으로 식별된 것들은 학습에 사용하기에는 너무 결측된 데이터가 많기 때문에, 해당 데이터를 사전에 제거합니다.

In [ ]:
df = df.drop(drop_cols, axis=1)
print(df.shape)

---

## 4. 결측값 대체
---
### 4.1 결측값 대체(Data Imputation)
너무 많은 결측치를 포함하는 결측 데이터 항목은 사전에 제거되었습니다. 남은 데이터 항목 중 데이터 결측치가 있을 경우, 빈 데이터 공간을 유의미한 데이터로 채워줘야 합니다.

데이터의 통계적인 특성을 고려하여 빈 데이터를 채우는 것을 결측값 대체(Data Imputation)이라고 합니다.

결측값 대체는 sklearn 라이브러리의 Iterative Imputer 함수를 이용해 수행합니다.

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
def imputation(df): 
    isna_stat = (df.isna().sum()/df.shape[0]).sort_values(ascending=True) 
    if isna_stat.max() > 0.0: 
       print('IterativeImputer를 이용해 데이터 결측값을 대체합니다.') 
       df = pd.DataFrame(IterativeImputer(random_state=0).fit_transform(df), columns = df.columns)  
    else: 
       print('결측값 대체를 할 필요가 없습니다.')
    return df

통계적 특성을 이용하므로, 숫자 데이터에 대해서만 결측값 대체를 수행할 수 있습니다. 

물론, 글자 데이터를 숫자 데이터로 카테고리화 해준 뒤에는 결측값 대체를 수행할 수 있습니다. 하지만 본 실습에서는 우선 숫자 데이터에 대해서만 결측값 대체를 수행해보겠습니다. 

토이 데이터셋 중 숫자 데이터 항목만 선택합니다.

In [ ]:
df_imputed = df.select_dtypes(include='number')
display(df_imputed)
df_imputed.isnull().sum()

숫자 데이터인 Age 데이터 항목에 177 개의 결측치가 있습니다. 사전에 정의한 imputation() 함수를 이용해 결측치 대체를 수행합니다.

### [TODO] 결측값 대체를 수행하기 위해 적합한 함수를 _______에 넣어보세요.

In [ ]:
df_imputed = _______(df_imputed)
display(df_imputed)
df_imputed.isnull().sum()

Age 데이터 항목의 모든 결측값을 성공적으로 대체하였습니다.

---

## 5. 이상치 전처리
---
### 5.1 이상치 검출 및 제거
현실 세계에는 많은 불확실성이 존재합니다. 특히, 데이터를 센서를 통해 획득할 때는 센서의 한계 때문에 이상치(outlier)가 기록되기도 합니다. 

이러한 데이터는 학습 성능을 크게 저하시키기 때문에, 이상치라는 것이 확실하다면 이를 사전이 제거해주는 전처리 과정을 거쳐야합니다.

데이터 이상치를 판별할 때 Z score를 고려하는 경우가 많습니다.

threshold_Z 변수는 사용자가 정의하는 임계값입니다. Z score가 3.0 이상일 경우, 이상치라고 판단하겠습니다.

### [TODO] 위 설명에서 요구하는대로 ____를 적절히 수정해보세요.

In [ ]:
from scipy import stats
threshold_Z = _____
def get_outliers(df, threshold_Z):
   Z_score = np.abs(stats.zscore(df)) 
   non_outliers =  (Z_score < threshold_Z).all(axis=1)
   outlier_index = non_outliers[non_outliers == False].index

   if len(outlier_index) != 0:
      print("이상치를 총 {}개 검출했습니다. 이상치를 갖는 데이터 행은 아래와 같습니다:".format(len(outlier_index)))
      print(outlier_index)
   else:
      print('현재 threshold_Z을 기반으로 검사했을 때 이상치를 찾지 못했습니다.') 
   return non_outliers

사전에 정의한 outliters() 함수를 이용해 이상치를 검출합니다.

In [ ]:
df = df_imputed
non_outliers = get_outliers(df, threshold_Z)

non_outliers 변수는 이상치가 아닌 정상 데이터의 행을 표시합니다. 이를 활용해 검출한 이상치를 제거해줍니다.

In [ ]:
df = df[non_outliers]
display(df)

69개의 데이터 행이 성공적으로 제거되었습니다. 따라서, 총 데이터 수가 891 개에서 822 개로 감소했습니다.

---

## 6. 실습 데이터 불러오기 및 전처리 알고리즘 적용
---

### 6.1 실습 데이터 불러오기
토이 데이터셋에서 적용했던 전처리 알고리즘을 실습 데이터에 순차적으로 적용해보겠습니다.

양극재 결정구조 분류 데이터셋을 불러옵니다. 

The Materials Project의 mp_api 를 사용하여 데이터를 다운받지 않는 이유는 다음과 같습니다:
- 개인 API를 모두 발급받아야 한다.
- 교육 과정동안 데이터를 매번 다운로드 받아야해서 번거롭다.
- 필수적인 데이터 전처리가 되어있지 않다.

위와 같은 이유로 mp_api를 이용하여 데이터를 미리 다운받아 실습하기 편한 csv 형태로 제공해드립니다.

pandas 라이브러리를 이용해 데이터를 불러옵니다.

In [ ]:
df = pd.read_csv('./data/lithium-ion batteries.csv') # df: dataframe의 줄임말
print(df)

In [ ]:
# display 함수를 이용하면 표 형태로 데이터 확인이 가능합니다.
display(df)

In [ ]:
# shape 기능을 이용하면 데이터의 차원을 확인할 수 있습니다.
print(df.shape)

총 339 개의 데이터와 11 개의 데이터 항목으로 구성된 데이터셋입니다.

---

### 6.2 데이터 자료형 확인

In [ ]:
plt.figure()
df.dtypes.value_counts().plot.pie(ylabel='')
plt.title('Data Types')
plt.show()

---

### 6.3 데이터 통계량 확인

실습 데이터에 대해서도 통계량을 확인해봅니다. 숫자 데이터인 int, float에 대해서만 통계량을 확인합니다.

In [ ]:
display(df.describe().T)

---

### 6.4 데이터 고유값 확인


In [ ]:
plt.figure()
df.nunique().plot.bar()
plt.title('Number of different values')
plt.show()

Materials ID, Formula, Spacegroup 세 데이터 항목은 글자 데이터이고, 그렇기 때문에 고유값이 많습니다.

Has Bandstructure는 참/거짓으로만 이뤄진 데이터이기 때문에 고유값이 두 개 입니다.

Crystal System은 monoclinic, orthorhombic, triclinic 세 가지 고유값을 가집니다.

---

### 6.5 결측 데이터 전처리

결측 데이터의 갯수와 비율을 확인해봅니다.

In [ ]:
isNaN = df.isna().sum().sort_values(ascending=True)
isNaN_ratio = isNaN/df.shape[0]
print("NaN 데이터 갯수: \n", isNaN)
print("\n")
print("NaN 데이터 비율: \n", isNaN_ratio)

실습 데이터에는 결측치가 존재하지 않는 것을 확인할 수 있습니다.

따라서, 결측치 대체는 자연스럽게 수행할 필요가 없습니다.

---

### 6.6 이상치 검출 및 제거

토이 데이터셋에 적용했던 방식 그대로, 이상치를 검출하고 제거해봅니다. 동일한 threshold_Z 값을 사용하겠습니다.

In [ ]:
df_imputed = df.select_dtypes(include='number')
non_outliers = get_outliers(df_imputed, threshold_Z)

In [ ]:
df = df[non_outliers]
display(df)

총 10 개의 데이터 엔트리가 이상치로 판단되어 제거되었습니다. 따라서, 339 개 데이터에서 329 개 데이터로 감소되었습니다.

---

## 7. 데이터 구분
---
### 7.1 정답 데이터 분리
전처리를 모두 마친 데이터셋에서 정답지로 삼을 데이터 항목을 분리합니다. 본 실습에서는 결정계인 'Crystal System'을 분리합니다.

### [TODO] 위 설명에서 요구하는대로 ____를 적절히 수정해보세요.

In [ ]:
target_col = '_______'
y = df[target_col]
display(y)

---

### 7.2 클래스 밸런스 확인
정답 데이터를 분류 문제(classification)로 학습하고자 할 때, 정답 데이터를 클래스(class)라고 부릅니다.

이때 클래스 간의 데이터 수가 적절히 비슷해야 학습 성능을 보장할 수 있다.

클래스 간의 불균형(Class Unbalance)이 너무 클 경우, 데이터를 더 획득하거나 기타 머신러닝 기법을 사용해야 한다.


In [ ]:
from yellowbrick.target import class_balance
class_balance(y)

---

### 7.3 학습, 테스트 데이터셋 구분
학습을 하기 전, 학습에 사용할 데이터와 검증하는데 사용할 데이터를 분리합니다.

실습 데이터의 크기가 작기 때문에, 테스트 데이터를 검증 데이터로 활용하겠습니다.

__학습 데이터를 7, 테스트 데이터를 3__ 의 비율로 구분합니다.

In [ ]:
X = df.drop(target_col, axis=1)
display(X)

random_seed 는 데이터를 구분할 때 랜덤 시드 숫자를 고정하기 위한 변수입니다.

데이터를 랜덤으로 섞은 후 두 데이터셋을 분리하는데, 이때 랜덤하게 데이터를 섞으면 매 실행마다 학습 결과가 달라지게 됩니다.

따라서, 랜덤하게 데이터를 섞되, 시드 숫자를 고정하여 다시 실행했을 때 똑같은 결과를 언제든지 재현할 수 있도록 합니다.

시드 숫자는 어떠한 숫자를 사용해도 괜찮습니다.

In [ ]:
from sklearn.model_selection import train_test_split
test_size = 0.3
random_seed = 42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, stratify=None,\
                                                           shuffle=True, random_state = random_seed)

In [ ]:
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)